# Prática: do full scan ao HNSW

Esta prática isola o comportamento interno do índice vetorial. Primeiro armazenaremos pontos sem construir HNSW e confirmaremos que eles continuam pesquisáveis. Depois habilitaremos o optimizer e compararemos uma consulta aproximada com o full scan solicitado por `exact=True`.

Os vetores são sintéticos e reproduzíveis. Eles não representam textos nem medem qualidade semântica. O exemplo espera um Qdrant acessível em `http://localhost:6333` e usa `qdrant-client==1.15.1` com `numpy`.

## 1. Conectando ao servidor

Usaremos outra coleção para não misturar este experimento com os chunks da primeira prática. A coleção será recriada sempre que o notebook rodar.

In [15]:
import os
import time

import numpy as np
from qdrant_client import QdrantClient, models

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "pratica-artigo-05-full-scan-hnsw"
VECTOR_NAME = "synthetic_dense"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
server = client.info()

print(f"Qdrant conectado: {QDRANT_URL}")
print(f"versão do servidor: {server.version}")
print(f"coleção da prática: {COLLECTION_NAME}")

Qdrant conectado: http://localhost:6333
versão do servidor: 1.15.4
coleção da prática: pratica-artigo-05-full-scan-hnsw


## 2. Gerando um conjunto vetorial controlado

Precisamos de dados suficientes para justificar um índice. Uma seed fixa produz sempre a mesma matriz, permitindo repetir a comparação sem depender de um modelo de embedding.

In [16]:
SEED = 42
POINTS_COUNT = 5_000
VECTOR_SIZE = 64
TOP_K = 10

rng = np.random.default_rng(SEED)
vectors = rng.normal(size=(POINTS_COUNT, VECTOR_SIZE)).astype(np.float32)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
query_vector = rng.normal(size=VECTOR_SIZE).astype(np.float32)
query_vector /= np.linalg.norm(query_vector)

print(f"seed: {SEED}")
print(f"pontos gerados: {POINTS_COUNT}")
print(f"dimensões por vetor: {VECTOR_SIZE}")
print(f"tamanho da matriz: {vectors.nbytes / 1024:.0f} KB")

seed: 42
pontos gerados: 5000
dimensões por vetor: 64
tamanho da matriz: 1250 KB


## 3. Criando a coleção sem indexação vetorial

`indexing_threshold = 0` desabilita a construção do HNSW. Os parâmetros do grafo já ficam definidos, mas só serão usados quando habilitarmos o optimizer.

In [17]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        VECTOR_NAME: models.VectorParams(
            size=VECTOR_SIZE,
            distance=models.Distance.COSINE,
        )
    },
    hnsw_config=models.HnswConfigDiff(
        m=8,
        ef_construct=32,
        max_indexing_threads=1,
        full_scan_threshold=10,
    ),
    optimizers_config=models.OptimizersConfigDiff(
        indexing_threshold=0,
        default_segment_number=2,
    ),
)

collection = client.get_collection(COLLECTION_NAME)

print(f"indexing_threshold: {collection.config.optimizer_config.indexing_threshold} KB")
print(f"HNSW configurado: m={collection.config.hnsw_config.m}")
print(f"pontos armazenados: {collection.points_count}")
print(f"vetores em índices: {collection.indexed_vectors_count}")

indexing_threshold: 0 KB
HNSW configurado: m=8
pontos armazenados: 0
vetores em índices: 0


## 4. Inserindo pontos que ainda não possuem HNSW

O `upsert` espera a aplicação dos pontos. A Count API fornece a quantidade lógica exata, enquanto `indexed_vectors_count` descreve aproximadamente o estado físico dos índices.

In [18]:
points = [
    models.PointStruct(
        id=point_id,
        vector={VECTOR_NAME: vector.tolist()},
    )
    for point_id, vector in enumerate(vectors)
]
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True,
)

exact_count = client.count(
    collection_name=COLLECTION_NAME,
    exact=True,
).count
collection = client.get_collection(COLLECTION_NAME)

print(f"pontos, contagem exata: {exact_count}")
print(f"pontos, estado físico aproximado: {collection.points_count}")
print(f"vetores em índices, valor aproximado: {collection.indexed_vectors_count}")
print(f"status da coleção: {collection.status.value}")

pontos, contagem exata: 5000
pontos, estado físico aproximado: 5000
vetores em índices, valor aproximado: 0
status da coleção: green


## 5. Consultando antes da construção do índice

A consulta normal continua funcionando. Como nenhum vetor está em HNSW, o Qdrant precisa comparar a query com os vetores armazenados nos segmentos plain.

In [19]:
def point_ids(response):
    return [point.id for point in response.points]

def print_ranking(title, response):
    print(title)
    for position, point in enumerate(response.points, start=1):
        print(f"{position:>2}. ponto {point.id:<4} | score={point.score:.4f}")

before_index = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector.tolist(),
    using=VECTOR_NAME,
    limit=TOP_K,
    with_payload=False,
    with_vectors=False,
)

print_ranking("candidatos antes do HNSW:", before_index)

candidatos antes do HNSW:
 1. ponto 3990 | score=0.4803
 2. ponto 2288 | score=0.4413
 3. ponto 4950 | score=0.4038
 4. ponto 1670 | score=0.4016
 5. ponto 1774 | score=0.3962
 6. ponto 1042 | score=0.3775
 7. ponto 3543 | score=0.3758
 8. ponto 3085 | score=0.3727
 9. ponto 1665 | score=0.3693
10. ponto 2758 | score=0.3585


## 6. Habilitando o optimizer

Um limite positivo permite que segmentos acima desse tamanho sejam reconstruídos com HNSW. A célula aguarda o estado estável sem imprimir o progresso intermediário, que varia conforme a máquina.

In [20]:
INDEXING_THRESHOLD_KB = 20
client.update_collection(
    collection_name=COLLECTION_NAME,
    optimizers_config=models.OptimizersConfigDiff(
        indexing_threshold=INDEXING_THRESHOLD_KB,
    ),
)

deadline = time.monotonic() + 120
stable_reads = 0
last_indexed = -1

while time.monotonic() < deadline:
    collection = client.get_collection(COLLECTION_NAME)
    indexed = collection.indexed_vectors_count or 0
    stable = indexed > 0 and indexed == last_indexed
    ready = (
        collection.status == models.CollectionStatus.GREEN
        and collection.optimizer_status == models.OptimizersStatusOneOf.OK
    )
    stable_reads = stable_reads + 1 if stable and ready else 0
    if stable_reads >= 2:
        break
    last_indexed = indexed
    time.sleep(0.5)
else:
    raise TimeoutError("O optimizer não concluiu a indexação em 120 segundos.")

exact_count = client.count(COLLECTION_NAME, exact=True).count

print(f"indexing_threshold: {collection.config.optimizer_config.indexing_threshold} KB")
print(f"pontos, contagem exata: {exact_count}")
print(f"vetores em índices, valor aproximado: {collection.indexed_vectors_count}")
print(f"status da coleção: {collection.status.value}")
print(f"status do optimizer: {collection.optimizer_status.value}")

indexing_threshold: 20 KB
pontos, contagem exata: 5000
vetores em índices, valor aproximado: 5000
status da coleção: green
status do optimizer: ok


## 7. Comparando busca aproximada e full scan

A consulta normal permite o uso do HNSW disponível. `exact=True` desabilita a busca aproximada e percorre todos os vetores elegíveis. A sobreposição abaixo mede apenas a qualidade do índice para esta query. Os valores de `m`, `ef_construct` e `hnsw_ef` foram escolhidos para tornar uma possível diferença visível, não como recomendação de produção. Os IDs aproximados podem variar conforme a versão do Qdrant e a organização física da coleção. O objetivo é observar o contraste, não reproduzir a mesma lista de IDs.

In [21]:
approximate = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector.tolist(),
    using=VECTOR_NAME,
    limit=TOP_K,
    search_params=models.SearchParams(hnsw_ef=40, exact=False),
    with_payload=False,
    with_vectors=False,
)
exact = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector.tolist(),
    using=VECTOR_NAME,
    limit=TOP_K,
    search_params=models.SearchParams(exact=True),
    with_payload=False,
    with_vectors=False,
)

approximate_ids = point_ids(approximate)
exact_ids = point_ids(exact)
shared = len(set(approximate_ids).intersection(exact_ids))

print("posição | aproximada | exata")
print("--------|------------|-------")
for position, (approximate_id, exact_id) in enumerate(
    zip(approximate_ids, exact_ids),
    start=1,
):
    print(f"{position:>7} | {approximate_id:>10} | {exact_id:>5}")

print(f"\nresultados compartilhados: {shared}/{TOP_K}")
print(f"recall@{TOP_K} nesta query: {shared / TOP_K:.2f}")
print(f"ranking antes do HNSW igual ao exato: {point_ids(before_index) == exact_ids}")

posição | aproximada | exata
--------|------------|-------
      1 |       3990 |  3990
      2 |       4950 |  2288
      3 |       1670 |  4950
      4 |       1774 |  1670
      5 |       1042 |  1774
      6 |       3085 |  1042
      7 |       1665 |  3543
      8 |       1270 |  3085
      9 |        253 |  1665
     10 |       2498 |  2758

resultados compartilhados: 7/10
recall@10 nesta query: 0.70
ranking antes do HNSW igual ao exato: True


A primeira consulta encontrou candidatos mesmo com zero vetores em HNSW, porque os pontos já estavam persistidos e disponíveis para full scan. Depois, o optimizer reconstruiu estruturas em segundo plano e `indexed_vectors_count` passou a indicar a presença de vetores indexados.

A comparação final não é uma avaliação de relevância e também não funciona como um `EXPLAIN` da consulta. Ela usa o full scan como referência para observar quanto o resultado aproximado se sobrepôs aos vizinhos exatos desta query.